# 11 — OOF stacking, calibration, and temporal consistency

**Group 119 — Henrik Remmen (541747)**  
**Kaggle team:** `[119] SINTEF Databreach`

This notebook is run only after the candidate model notebooks have created both an OOF artifact and a test submission.

It performs four controlled operations:

1. compare base models on identical chronological folds;
2. build a conservative greedy probability blend;
3. cross-fit a per-generator logistic stacker forward in time;
4. test small amounts of horizon smoothing and overlap consistency on OOF predictions.

No public leaderboard score is used to choose weights. Models without matching OOF predictions are excluded, even if their test submission exists.

> Uses up to 12 parallel CPU jobs. Run after notebooks 05–10.


## 1. Load project structure and available prediction pairs

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from joblib import Parallel, delayed
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.sintef_pipeline import (
    HORIZON,
    RANDOM_SEED,
    build_project_data,
    make_submission,
    micro_auc,
)

sns.set_theme(style="whitegrid", context="notebook")
data = build_project_data(ROOT)

ARTIFACT_DIR = ROOT / "artifacts" / "predictions"
SUBMISSION_DIR = ROOT / "submissions"
N_JOBS = min(12, os.cpu_count() or 1)

CANDIDATES = {
    "Improved logistic regression": (
        "logreg_improved_oof.npz",
        "logreg_improved.csv",
    ),
    "Improved LightGBM": (
        "lgbm_improved_oof.npz",
        "lgbm_improved.csv",
    ),
    "CatBoost per generator": (
        "catboost_per_generator_oof.npz",
        "catboost_per_generator.csv",
    ),
    "Shared CatBoost": (
        "catboost_shared_oof.npz",
        "catboost_shared.csv",
    ),
    "XGBoost per generator": (
        "xgboost_per_generator_oof.npz",
        "xgboost_per_generator.csv",
    ),
    "EBM per generator": (
        "ebm_per_generator_oof.npz",
        "ebm_per_generator.csv",
    ),
    "Multi-output ExtraTrees": (
        "extratrees_multioutput_oof.npz",
        "extratrees_multioutput.csv",
    ),
    "Multi-task TCN": (
        "tcn_multitask_oof.npz",
        "tcn_multitask.csv",
    ),
}

print("Project root:", ROOT.resolve())
print(f"Parallel CPU jobs: {N_JOBS}/{os.cpu_count()} logical cores")


In [ ]:
def submission_to_cube(path):
    submission = pd.read_csv(path)
    if not submission["Run No"].equals(data.sample_submission["Run No"]):
        raise ValueError(f"Run No mismatch in {path.name}")

    cube = np.stack([
        submission[
            [f"result_committed_{unit}_t{hour}" for hour in range(HORIZON)]
        ].to_numpy(dtype=np.float32)
        for unit in data.units
    ], axis=1)
    expected = (data.n_test_cases, len(data.units), HORIZON)
    if cube.shape != expected or not np.isfinite(cube).all():
        raise ValueError(f"Invalid submission cube in {path.name}: {cube.shape}")
    return cube

oof_predictions = {}
test_predictions = {}
missing_pairs = []

for name, (oof_file, submission_file) in CANDIDATES.items():
    oof_path = ARTIFACT_DIR / oof_file
    submission_path = SUBMISSION_DIR / submission_file
    if not oof_path.exists() or not submission_path.exists():
        missing_pairs.append(name)
        continue

    oof = np.load(oof_path)["oof"].astype(np.float32)
    if oof.shape != data.target_cube.shape:
        raise ValueError(f"Unexpected OOF shape for {name}: {oof.shape}")
    oof_predictions[name] = oof
    test_predictions[name] = submission_to_cube(submission_path)

print("Available paired models:", list(oof_predictions))
print("Skipped because OOF or submission is missing:", missing_pairs)
if len(oof_predictions) < 2:
    raise RuntimeError(
        "Run at least two candidate model notebooks before stacking."
    )

MODEL_NAMES = list(oof_predictions)


## 2. Base-model comparison and diversity

All scores and correlations below use only cases that were genuinely out of fold.

In [ ]:
def fold_score_rows(prediction, model_name):
    rows = []
    for fold in data.folds:
        case_ids = fold["validation_cases"]
        rows.append({
            "model": model_name,
            "validation_year": fold["year"],
            "micro_auc": micro_auc(
                data.target_cube[case_ids],
                prediction[case_ids],
            ),
        })
    return rows

base_score_groups = Parallel(n_jobs=N_JOBS, prefer="threads")(
    delayed(fold_score_rows)(prediction, name)
    for name, prediction in oof_predictions.items()
)
base_score_rows = [
    row for group in base_score_groups for row in group
]

base_fold_scores = pd.DataFrame(base_score_rows)
base_summary = (
    base_fold_scores.groupby("model")
    .agg(
        mean_auc=("micro_auc", "mean"),
        worst_auc=("micro_auc", "min"),
        fold_std=("micro_auc", "std"),
    )
    .sort_values(["mean_auc", "worst_auc"], ascending=False)
)
display(base_summary.style.format("{:.6f}"))

validation_case_ids = np.concatenate([
    fold["validation_cases"] for fold in data.folds
])
flat_predictions = {
    name: prediction[validation_case_ids].ravel()
    for name, prediction in oof_predictions.items()
}
prediction_correlation = pd.DataFrame(flat_predictions).corr()

flat_targets = data.target_cube[validation_case_ids].ravel()
flat_residuals = {
    name: flat_targets-values
    for name, values in flat_predictions.items()
}
residual_correlation = pd.DataFrame(flat_residuals).corr()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(
    prediction_correlation,
    vmin=0,
    vmax=1,
    cmap="viridis",
    annot=True,
    fmt=".3f",
    ax=axes[0],
)
axes[0].set_title("OOF prediction correlation")
sns.heatmap(
    residual_correlation,
    vmin=0,
    vmax=1,
    cmap="magma",
    annot=True,
    fmt=".3f",
    ax=axes[1],
)
axes[1].set_title("OOF residual correlation")
plt.tight_layout()
plt.show()


## 3. Conservative greedy blend

Starting from the strongest mean-OOF model, a new model is added only if mean AUC improves and worst-fold AUC does not fall by more than 0.001. This keeps the search space small.

In [ ]:
def score_prediction(prediction, years=(2020, 2021, 2022)):
    fold_scores = {}
    for fold in data.folds:
        if fold["year"] not in years:
            continue
        case_ids = fold["validation_cases"]
        fold_scores[fold["year"]] = micro_auc(
            data.target_cube[case_ids],
            prediction[case_ids],
        )
    values = list(fold_scores.values())
    return {
        "mean_auc": float(np.mean(values)),
        "worst_auc": float(np.min(values)),
        **{f"auc_{year}": score for year, score in fold_scores.items()},
    }

def evaluate_blend_trial(name, alpha, current_oof):
    trial_prediction = (
        (1-alpha)*current_oof + alpha*oof_predictions[name]
    )
    return {
        "name": name,
        "alpha": float(alpha),
        **score_prediction(trial_prediction),
    }

ALPHAS = np.arange(0.05, 0.55, 0.05)
start_name = base_summary.index[0]
greedy_oof = oof_predictions[start_name].copy()
greedy_test = test_predictions[start_name].copy()
greedy_weights = {start_name: 1.0}
remaining = [name for name in MODEL_NAMES if name != start_name]
greedy_history = [{
    "step": 0,
    "added_model": start_name,
    "added_weight": 1.0,
    **score_prediction(greedy_oof),
}]

for step in range(1, min(4, len(MODEL_NAMES))):
    current_score = score_prediction(greedy_oof)
    trials = Parallel(n_jobs=N_JOBS, prefer="threads")(
        delayed(evaluate_blend_trial)(name, alpha, greedy_oof)
        for name in remaining
        for alpha in ALPHAS
    )

    best_trial = max(
        trials,
        key=lambda row: (row["mean_auc"], row["worst_auc"]),
    )
    mean_improved = (
        best_trial["mean_auc"] > current_score["mean_auc"] + 1e-5
    )
    worst_acceptable = (
        best_trial["worst_auc"] >= current_score["worst_auc"] - 0.001
    )
    if not (mean_improved and worst_acceptable):
        break

    alpha = best_trial["alpha"]
    added_name = best_trial["name"]
    greedy_oof = (
        (1-alpha)*greedy_oof
        + alpha*oof_predictions[added_name]
    )
    greedy_test = (
        (1-alpha)*greedy_test
        + alpha*test_predictions[added_name]
    )
    greedy_weights = {
        name: weight*(1-alpha)
        for name, weight in greedy_weights.items()
    }
    greedy_weights[added_name] = alpha
    remaining.remove(added_name)
    greedy_history.append({
        "step": step,
        "added_model": added_name,
        "added_weight": alpha,
        **score_prediction(greedy_oof),
    })

greedy_history = pd.DataFrame(greedy_history)
print("Final greedy weights:")
display(pd.Series(greedy_weights, name="weight").sort_values(ascending=False))
display(greedy_history.style.format({
    "added_weight": "{:.2f}",
    "mean_auc": "{:.6f}",
    "worst_auc": "{:.6f}",
}))

greedy_path, greedy_submission = make_submission(
    data,
    greedy_test,
    "ensemble_greedy_oof_blend.csv",
)
print("Saved:", greedy_path)


## 4. Forward-time logistic stacker

For validation year 2021, the meta-model is trained only on 2020 OOF predictions. For validation year 2022, it is trained only on 2020–2021 OOF predictions. A separate sigmoid stacker is fitted for each generator, which also calibrates probability scales across units.

The final stacker is fitted on 2020–2022 OOF predictions and applied to the full-data base-model submissions.

In [ ]:
def safe_logit(probabilities, epsilon=1e-5):
    probabilities = np.clip(probabilities, epsilon, 1-epsilon)
    return np.log(probabilities/(1-probabilities))

def meta_features(prediction_dict, case_ids, unit_index):
    return np.column_stack([
        safe_logit(prediction_dict[name][case_ids, unit_index, :]).reshape(-1)
        for name in MODEL_NAMES
    ])

def make_meta_model():
    return Pipeline([
        ("scale", StandardScaler()),
        ("classifier", LogisticRegression(
            C=0.3,
            solver="lbfgs",
            max_iter=500,
            random_state=RANDOM_SEED,
        )),
    ])

def fit_forward_unit(unit_index, train_cases, validation_cases):
    X_meta_train = meta_features(
        oof_predictions, train_cases, unit_index
    )
    y_meta_train = (
        data.target_cube[train_cases, unit_index, :].reshape(-1)
    )
    X_meta_val = meta_features(
        oof_predictions, validation_cases, unit_index
    )
    model = make_meta_model()
    model.fit(X_meta_train, y_meta_train)
    prediction = model.predict_proba(X_meta_val)[:, 1]
    return unit_index, prediction.reshape(len(validation_cases), HORIZON)

stack_oof = np.full(data.target_cube.shape, np.nan, dtype=np.float32)
stack_fold_rows = []

for validation_year in (2021, 2022):
    meta_train_cases = np.concatenate([
        fold["validation_cases"]
        for fold in data.folds
        if fold["year"] < validation_year
    ])
    meta_val_cases = next(
        fold["validation_cases"]
        for fold in data.folds
        if fold["year"] == validation_year
    )
    fold_prediction = np.empty(
        (len(meta_val_cases), len(data.units), HORIZON),
        dtype=np.float32,
    )

    with threadpool_limits(limits=1):
        unit_results = Parallel(n_jobs=N_JOBS, prefer="threads")(
            delayed(fit_forward_unit)(
                unit_index, meta_train_cases, meta_val_cases
            )
            for unit_index in range(len(data.units))
        )
    for unit_index, prediction in unit_results:
        fold_prediction[:, unit_index, :] = prediction

    stack_oof[meta_val_cases] = fold_prediction
    score = micro_auc(
        data.target_cube[meta_val_cases],
        fold_prediction,
    )
    stack_fold_rows.append({
        "validation_year": validation_year,
        "micro_auc": score,
    })
    print(f"Forward stack | {validation_year} | AUC={score:.6f}")

stack_fold_scores = pd.DataFrame(stack_fold_rows)
display(stack_fold_scores.style.format({"micro_auc": "{:.6f}"}))
print("Forward-stack mean:", stack_fold_scores["micro_auc"].mean())
print("Forward-stack worst:", stack_fold_scores["micro_auc"].min())


In [ ]:
# FAST FINAL META-FIT — logistic stackers only, no base-model retraining
all_meta_cases = validation_case_ids
stack_test = np.empty(
    (data.n_test_cases, len(data.units), HORIZON),
    dtype=np.float32,
)
final_meta_coefficients = []

def fit_final_unit(unit_index):
    unit = data.units[unit_index]
    X_meta_train = meta_features(
        oof_predictions,
        all_meta_cases,
        unit_index,
    )
    y_meta_train = (
        data.target_cube[all_meta_cases, unit_index, :]
        .reshape(-1)
    )
    X_meta_test = np.column_stack([
        safe_logit(test_predictions[name][:, unit_index, :]).reshape(-1)
        for name in MODEL_NAMES
    ])

    model = make_meta_model()
    model.fit(X_meta_train, y_meta_train)
    prediction = model.predict_proba(X_meta_test)[:, 1]
    coefficients = model.named_steps["classifier"].coef_[0]
    coefficient_rows = []
    for model_name, coefficient in zip(MODEL_NAMES, coefficients):
        coefficient_rows.append({
            "unit": unit,
            "base_model": model_name,
            "coefficient_on_scaled_logit": coefficient,
        })
    return (
        unit_index,
        prediction.reshape(data.n_test_cases, HORIZON),
        coefficient_rows,
    )

with threadpool_limits(limits=1):
    final_unit_results = Parallel(n_jobs=N_JOBS, prefer="threads")(
        delayed(fit_final_unit)(unit_index)
        for unit_index in range(len(data.units))
    )
for unit_index, prediction, coefficient_rows in final_unit_results:
    stack_test[:, unit_index, :] = prediction
    final_meta_coefficients.extend(coefficient_rows)

stack_path, stack_submission = make_submission(
    data,
    stack_test,
    "ensemble_logistic_stack.csv",
)
print("Saved:", stack_path)
display(
    pd.DataFrame(final_meta_coefficients)
    .pivot(index="unit", columns="base_model", values="coefficient_on_scaled_logit")
    .style.format("{:.3f}")
)


## 5. Select on common forward years and test temporal consistency

Only 2021–2022 can be compared fairly with the forward stack. The post-processing grid is deliberately small. Smoothing is performed in logit space so extreme probabilities remain well behaved.

In [ ]:
common_years = (2021, 2022)
common_case_ids = np.concatenate([
    fold["validation_cases"]
    for fold in data.folds
    if fold["year"] in common_years
])

comparison_candidates = {
    "Greedy blend": (greedy_oof, greedy_test),
    "Forward logistic stack": (stack_oof, stack_test),
}
for name in MODEL_NAMES:
    comparison_candidates[f"Base: {name}"] = (
        oof_predictions[name],
        test_predictions[name],
    )

def evaluate_comparison_candidate(name, oof_prediction):
    metrics = score_prediction(oof_prediction, years=common_years)
    return {"candidate": name, **metrics}

comparison_rows = Parallel(n_jobs=N_JOBS, prefer="threads")(
    delayed(evaluate_comparison_candidate)(name, oof_prediction)
    for name, (oof_prediction, _) in comparison_candidates.items()
)

common_comparison = (
    pd.DataFrame(comparison_rows)
    .sort_values(["mean_auc", "worst_auc"], ascending=False)
    .reset_index(drop=True)
)
display(common_comparison.style.format({
    "mean_auc": "{:.6f}",
    "worst_auc": "{:.6f}",
}))

SELECTED_NAME = common_comparison.loc[0, "candidate"]
selected_oof, selected_test = comparison_candidates[SELECTED_NAME]
print("Selected before post-processing:", SELECTED_NAME)


In [ ]:
def sigmoid(values):
    return 1/(1+np.exp(-values))

def smooth_horizon_logits(prediction, alpha, window):
    if alpha == 0:
        return prediction.copy()
    logits = safe_logit(prediction)
    pad = window//2
    padded = np.pad(
        logits,
        ((0, 0), (0, 0), (pad, pad)),
        mode="edge",
    )
    rolling = sum(
        padded[:, :, offset:offset+logits.shape[2]]
        for offset in range(window)
    )/window
    return sigmoid((1-alpha)*logits + alpha*rolling).astype(np.float32)

def overlap_consistency(prediction, starts, alpha):
    if alpha == 0:
        return prediction.copy()
    starts = pd.DatetimeIndex(starts)
    timestamps = pd.DatetimeIndex(
        starts.repeat(HORIZON)
        + pd.to_timedelta(
            np.tile(np.arange(HORIZON), len(starts)),
            unit="h",
        )
    )
    logits = safe_logit(prediction)
    adjusted = np.empty_like(logits)

    for unit_index in range(logits.shape[1]):
        flat = logits[:, unit_index, :].reshape(-1)
        timestamp_mean = (
            pd.Series(flat)
            .groupby(timestamps)
            .transform("mean")
            .to_numpy()
        )
        combined = (1-alpha)*flat + alpha*timestamp_mean
        adjusted[:, unit_index, :] = combined.reshape(
            len(starts), HORIZON
        )
    return sigmoid(adjusted).astype(np.float32)


In [ ]:
# FAST OOF SEARCH — no model fitting
base_common = selected_oof[common_case_ids]
truth_common = data.target_cube[common_case_ids]
common_starts = data.train_cases.iloc[common_case_ids]["starttime"]

def evaluate_smoothing(window, alpha):
    candidate = smooth_horizon_logits(base_common, alpha, window)
    return {
            "window": window,
            "alpha": alpha,
            "micro_auc": micro_auc(truth_common, candidate),
    }

smoothing_rows = Parallel(n_jobs=N_JOBS, prefer="threads")(
    delayed(evaluate_smoothing)(window, alpha)
    for window in (3, 5, 9)
    for alpha in (0.0, 0.10, 0.20, 0.30)
)

smoothing_results = (
    pd.DataFrame(smoothing_rows)
    .sort_values("micro_auc", ascending=False)
    .reset_index(drop=True)
)
BEST_WINDOW = int(smoothing_results.loc[0, "window"])
BEST_SMOOTH_ALPHA = float(smoothing_results.loc[0, "alpha"])
smoothed_common = smooth_horizon_logits(
    base_common,
    BEST_SMOOTH_ALPHA,
    BEST_WINDOW,
)

overlap_rows = []
for alpha in (0.0, 0.10, 0.25, 0.50):
    candidate = overlap_consistency(
        smoothed_common,
        common_starts,
        alpha,
    )
    overlap_rows.append({
        "alpha": alpha,
        "micro_auc": micro_auc(truth_common, candidate),
    })

overlap_results = (
    pd.DataFrame(overlap_rows)
    .sort_values("micro_auc", ascending=False)
    .reset_index(drop=True)
)
BEST_OVERLAP_ALPHA = float(overlap_results.loc[0, "alpha"])

print("Selected candidate:", SELECTED_NAME)
print("Best horizon window:", BEST_WINDOW)
print("Best horizon alpha:", BEST_SMOOTH_ALPHA)
print("Best overlap alpha:", BEST_OVERLAP_ALPHA)
display(smoothing_results.head(10).style.format({"micro_auc": "{:.6f}"}))
display(overlap_results.style.format({"micro_auc": "{:.6f}"}))


## 6. Create the OOF-selected final post-processed submission

If the best smoothing or overlap weight is zero, that operation is automatically skipped.

In [ ]:
# FAST SUBMISSION CELL — no base-model fitting
processed_test = smooth_horizon_logits(
    selected_test,
    BEST_SMOOTH_ALPHA,
    BEST_WINDOW,
)
processed_test = overlap_consistency(
    processed_test,
    data.test_cases["starttime"],
    BEST_OVERLAP_ALPHA,
)

processed_path, processed_submission = make_submission(
    data,
    processed_test,
    "ensemble_oof_selected_temporal.csv",
)
print("Saved:", processed_path)
display(processed_submission.iloc[:3, :8])

ensemble_manifest = pd.DataFrame([
    {
        "selected_candidate": SELECTED_NAME,
        "greedy_weights": repr(greedy_weights),
        "horizon_window": BEST_WINDOW,
        "horizon_alpha": BEST_SMOOTH_ALPHA,
        "overlap_alpha": BEST_OVERLAP_ALPHA,
        "available_models": repr(MODEL_NAMES),
    }
])
manifest_path = ARTIFACT_DIR / "ensemble_manifest.csv"
ensemble_manifest.to_csv(manifest_path, index=False)
print("Saved:", manifest_path)


## Upload order and descriptions

Upload only after checking the local tables:

1. `ensemble_greedy_oof_blend.csv`  
   Description: `Greedy OOF blend – mean/worst chronological promotion rule`

2. `ensemble_logistic_stack.csv`  
   Description: `Forward OOF logistic stack – per-unit probability calibration`

3. `ensemble_oof_selected_temporal.csv`  
   Description: `OOF-selected ensemble – temporal and overlap consistency`

Do not include the older LogReg–LightGBM blend in the learned stack unless matching OOF artifacts are regenerated. A test submission without OOF predictions cannot be assigned a leakage-safe weight.
